## **DimUser**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys

project_path = os.path.join(os.getcwd(), '..','..')
sys.path.append(project_path)
from utils.transformations import reusable

## **AutoLoader**

In [0]:
df_user = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimUser/checkpoint")\
                .load("abfss://bronze@storageazureproject03.dfs.core.windows.net/DimUser")

In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))
display(df_user, checkpointLocation="abfss://silver@storageazureproject03.dfs.core.windows.net/DimUser/display_checkpoint_v3")

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
print(df_user.columns)

In [0]:
df_user.writeStream.format("delta")\
                    .outputMode("append")\
                    .option("checkpointLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimUser/checkpoint")\
                    .trigger(once=True)\
                    .option("path","abfss://silver@storageazureproject03.dfs.core.windows.net/DimUser/data")\
                    .toTable("spotify_cata.silver.DimUser")

# **DimArtist**

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimArtist/checkpoint")\
                .load("abfss://bronze@storageazureproject03.dfs.core.windows.net/DimArtist")

In [0]:
df_artist_obj = reusable()

df_artist = df_artist_obj.dropColumns(df_artist,['_rescued_data'])
df_artist = df_artist.dropDuplicates(['artist_id'])

In [0]:
print(df_artist.columns)

In [0]:
df_artist.writeStream.format("delta")\
                    .outputMode("append")\
                    .option("checkpointLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimArtist/checkpoint")\
                    .trigger(once=True)\
                    .option("path", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimArtist/data")\
                    .toTable("spotify_cata.silver.DimArtist")

# **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimTrack/checkpoint")\
                .load("abfss://bronze@storageazureproject03.dfs.core.windows.net/DimTrack")

In [0]:
df_track = df_track.withColumn("durationFlag", when(col("duration_sec") < 150, "low")\
                                                .when(col("duration_sec") > 300, "medium")\
                                                .otherwise("high"))

In [0]:
df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"),'-',' '))

In [0]:
df_track_obj = reusable()

df_track = df_track_obj.dropColumns(df_track,['_rescued_data'])

In [0]:
print(df_track.columns)

In [0]:
df_track.writeStream.format("delta")\
                    .outputMode("append")\
                    .option("checkpointLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimTrack/checkpoint")\
                    .trigger(once=True)\
                    .option("path", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimTrack/data")\
                    .toTable("spotify_cata.silver.DimTrack")

# **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimDate/checkpoint")\
                .load("abfss://bronze@storageazureproject03.dfs.core.windows.net/DimDate")

In [0]:
df_date_obj = reusable().dropColumns(df_date,['_rescued_data'])

In [0]:
print(df_date_obj.columns)

In [0]:
df_date.writeStream.format("delta")\
                    .outputMode("append")\
                    .option("checkpointLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimDate/checkpoint")\
                    .trigger(once=True)\
                    .option("path", "abfss://silver@storageazureproject03.dfs.core.windows.net/DimDate/data")\
                    .toTable("spotify_cata.silver.DimDate")

# **FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/FactStream/checkpoint")\
                .load("abfss://bronze@storageazureproject03.dfs.core.windows.net/FactStream")

In [0]:
df_fact_obj = reusable().dropColumns(df_fact,['_rescued_data'])
print(df_fact_obj.columns)

In [0]:
df_fact.writeStream.format("delta")\
                    .outputMode("append")\
                    .option("checkpointLocation", "abfss://silver@storageazureproject03.dfs.core.windows.net/FactStream/checkpoint")\
                    .trigger(once=True)\
                    .option("path", "abfss://silver@storageazureproject03.dfs.core.windows.net/FactStream/data")\
                    .toTable("spotify_cata.silver.FactStream")

# **TESTING**

In [0]:
%sql
select * from spotify_cata.gold.dimtrack
where `__END_AT` is not null

In [0]:
%sql
select * from spotify_cata.gold.dimtrack
where track_id in (46,5)

In [0]:
%sql
SELECT COUNT(*) FROM spotify_cata.gold.dimuser;
SELECT COUNT(*) FROM spotify_cata.gold.dimtrack;
SELECT COUNT(*) FROM spotify_cata.gold.dimdate;
SELECT COUNT(*) FROM spotify_cata.gold.factstream;

In [0]:
%sql
SELECT 'dimuser' AS table_name, COUNT(*) AS row_count FROM spotify_cata.gold.dimuser
UNION ALL
SELECT 'dimtrack', COUNT(*) FROM spotify_cata.gold.dimtrack
UNION ALL
SELECT 'dimdate', COUNT(*) FROM spotify_cata.gold.dimdate
UNION ALL
SELECT 'factstream', COUNT(*) FROM spotify_cata.gold.factstream